# Qwen3-4B interviewer fine-tune, optimized for FREE Colab T4
Profile + Level + Topic -> one interview question.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Upload `by_profile_level_topic.json` to `/content`.
If pip asks you to restart the session after the install cell, restart and continue from the next cell.

In [ ]:
!pip install -q unsloth

In [ ]:
# Mount Drive FIRST: checkpoints go here so a Colab disconnect doesn't lose progress.
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/qwen3-interviewer"   # checkpoints + final adapter
DATA_PATH = "/content/by_profile_level_topic.json"

In [ ]:
import json, random, gc
from collections import Counter

PER_GROUP_CAP = 10     # max questions per (profile, level, topic) group
MAX_SAMPLES = 20000    # ~30-60 min on T4. Lower to 10000 for a quick test run.
SEED = 42
random.seed(SEED)

SYSTEM = "You are a technical interviewer. Ask exactly one interview question that matches the requested profile, experience level and topic."

def build_user(profile, level, topic):
    return f"Profile: {profile}\nLevel: {level}\nTopic: {topic}\n\nAsk one interview question."

with open(DATA_PATH, encoding="utf-8") as f:
    raw = json.load(f)

samples = []
for profile, levels in raw.items():
    for level, topics in levels.items():
        for topic, payload in topics.items():
            seen, qs = set(), []
            for item in payload["questions"]:
                q = " ".join(item["question"].split())
                if 15 <= len(q) <= 400 and q.lower() not in seen:   # dedupe within group, drop junk
                    seen.add(q.lower())
                    qs.append(q)
            random.shuffle(qs)
            samples += [(profile, level, topic, q) for q in qs[:PER_GROUP_CAP]]

del raw; gc.collect()          # free RAM (free Colab has ~12GB)
random.shuffle(samples)
samples = samples[:MAX_SAMPLES]
print("samples used:", len(samples))
print(Counter(s[0] for s in samples).most_common())

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_LEN = 160   # prompts + question are short; small length = fast + low VRAM

model, tok = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit",   # pre-quantized: faster download, less RAM
    max_seq_length=MAX_LEN,
    dtype=None,            # auto -> fp16 on T4
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,        # 0 is fastest in Unsloth
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",   # big VRAM saving
    random_state=SEED,
)

In [ ]:
# Tokenize ourselves and mask the prompt so loss is only on the question.
# Qwen3 non-thinking format (empty think block) so the model never wastes tokens "thinking".
from datasets import Dataset

def encode(ex):
    profile, level, topic, q = ex
    prompt = (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{build_user(profile, level, topic)}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(q + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (p + c)[:MAX_LEN]
    labels = ([-100] * len(p) + c)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

ds = Dataset.from_list([encode(s) for s in samples])
lens = [len(x) for x in ds["input_ids"]]
print(f"token length: mean={sum(lens)/len(lens):.0f} max={max(lens)}")
ds = ds.train_test_split(test_size=0.01, seed=SEED)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=16,     # short sequences -> fits easily; try 32 w/ grad_accum 1 if VRAM allows
    gradient_accumulation_steps=2,      # effective batch 32
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=20,
    weight_decay=0.01,
    optim="adamw_8bit",
    fp16=True, bf16=False,              # T4: fp16 only
    group_by_length=True,               # batches similar lengths -> less padding -> faster
    logging_steps=10,
    eval_strategy="steps", eval_steps=200,
    per_device_eval_batch_size=16,
    save_strategy="steps", save_steps=100, save_total_limit=2,   # frequent saves for disconnects
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["test"],
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
)

In [ ]:
# If the session disconnected earlier: re-run all cells above, then this cell resumes automatically.
import os, glob
has_ckpt = bool(glob.glob(os.path.join(OUT_DIR, "checkpoint-*")))
trainer.train(resume_from_checkpoint=True if has_ckpt else None)

In [ ]:
model.save_pretrained(f"{OUT_DIR}/final-lora")
tok.save_pretrained(f"{OUT_DIR}/final-lora")
print("saved to", f"{OUT_DIR}/final-lora")

In [ ]:
# Test it
FastLanguageModel.for_inference(model)

def ask(profile, level, topic, n=3, temperature=0.8):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": build_user(profile, level, topic)}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=temperature,
                         top_p=0.9, num_return_sequences=n, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

for q in ask("Backend", "Architect/Principal", "gRPC"):
    print("-", q)

## Optional exports
Uncomment one. GGUF is for Ollama / llama.cpp; merged_16bit is for vLLM / HF.

In [ ]:
# model.save_pretrained_merged(f"{OUT_DIR}/merged-16bit", tok, save_method="merged_16bit")
# model.save_pretrained_gguf(f"{OUT_DIR}/gguf", tok, quantization_method="q4_k_m")